In [1]:
import os
import pandas as pd

In [14]:
def determine_location(availability):
    """Maps availability markers to location classifications."""
    if pd.isna(availability):
        return "in person"  # Default fallback if missing
    avail_str = str(availability).strip()
    if avail_str == "◯":
        return "both"
    elif avail_str.lower() in ["online", "オン"]:
        return "online"
    else:
        return avail_str.lower()


def consolidate_blocks(df, is_actual=False):
    """Groups continuous time blocks for the same person, day, location, and approval status."""
    if df.empty:
        return df

    # Work with copy to avoid slice warnings
    df = df.copy()

    # Determine columns based on file type
    time_start_col = "actual_start_time" if is_actual else "proposed_start_time"
    time_end_col = "actual_end_time" if is_actual else "proposed_end_time"
    status_col = "is_approved" if not is_actual else "report"

    # 1. Create solid datetime columns for sorting and math
    df["dt_start"] = pd.to_datetime(
        df["date"] + " " + df[time_start_col].fillna(""), errors="coerce"
    )
    df["dt_end"] = pd.to_datetime(
        df["date"] + " " + df[time_end_col].fillna(""), errors="coerce"
    )

    # Drop entries that didn't parse correctly
    df = df.dropna(subset=["dt_start", "dt_end"])

    # 2. Sort to ensure chronological order per student
    df = df.sort_values(by=["student_name", "dt_start"]).reset_index(drop=True)

    # 3. Identify consecutive blocks: New group if there's a time gap OR status/location changes
    # A gap is when the current start time does not match the previous row's end time
    is_new_student = df["student_name"] != df["student_name"].shift()
    is_time_gap = df["dt_start"] != df["dt_end"].shift()
    is_status_change = df[status_col] != df[status_col].shift()

    # If parsing schedule, track location changes too
    if not is_actual:
        is_loc_change = df["location"] != df["location"].shift()
        group_condition = (
            is_new_student | is_time_gap | is_status_change | is_loc_change
        )
    else:
        group_condition = is_new_student | is_time_gap | is_status_change

    df["session_id"] = group_condition.cumsum()

    # 4. Aggregate to find the boundaries of the continuous block
    agg_dict = {
        "student_name": "first",
        "date": "first",
        "dt_start": "min",
        "dt_end": "max",
    }
    if not is_actual:
        agg_dict.update(
            {"location": "first", "availability": "first", "is_approved": "first"}
        )
    else:
        agg_dict.update({"report": "first"})

    consolidated = df.groupby("session_id").agg(agg_dict).reset_index(drop=True)
    return consolidated


def merge_schedules_and_reports(csv_path, xlsx_path, output_path):
    # 1. Load data
    df_csv = pd.read_csv(csv_path)
    df_xlsx = (
        pd.read_csv(xlsx_path)
        if xlsx_path.endswith(".csv")
        else pd.read_excel(xlsx_path)
    )

    # 2. Standardize column names
    df_csv = df_csv.rename(
        columns={
            "proposed_start": "proposed_start_time",
            "proposed_end": "proposed_end_time",
            "block_start_time": "proposed_start_time",
            "block_end_time": "proposed_end_time",
        }
    )

    df_xlsx = df_xlsx.rename(
        columns={
            "student": "student_name",
            "block_start": "actual_start_time",
            "block_end": "actual_end_time",
            "block_start_time": "actual_start_time",
            "block_end_time": "actual_end_time",
        }
    )

    # Clean string fields
    for df in [df_csv, df_xlsx]:
        df["date"] = df["date"].astype(str).str.strip()
        df["student_name"] = df["student_name"].astype(str).str.strip()

    # Apply requirement #1: Derive location logic for schedule
    df_csv["location"] = df_csv["availability"].apply(determine_location)

    # Apply requirement #2: Consolidate continuous chunks independently
    df_csv_grouped = consolidate_blocks(df_csv, is_actual=False)
    df_xlsx_grouped = consolidate_blocks(df_xlsx, is_actual=True)

    # 3. Merge the consolidated timelines together
    merged_df = pd.merge(
        df_csv_grouped,
        df_xlsx_grouped,
        on=["student_name", "date"],
        how="outer",
        suffixes=("_prop", "_act"),
    )

    # 4. Construct output schema
    final_df = pd.DataFrame()
    final_df["student"] = merged_df["student_name"]

    # Pull location from schedule; fallback to "in person" if only report exists
    final_df["location"] = merged_df["location"].fillna("in person")

    # Pass times cleanly
    final_df["proposed_start"] = merged_df["dt_start_prop"]
    final_df["proposed_end"] = merged_df["dt_end_prop"]

    final_df["stats"] = merged_df["is_approved"].fillna("")
    final_df["actual_start"] = merged_df["dt_start_act"]
    final_df["actual_end"] = merged_df["dt_end_act"]
    final_df["work_report"] = merged_df["report"].fillna("")
    final_df["staff_comments"] = ""  # Initialized empty placeholder

    # Drop lines where everything is empty/null
    final_df = final_df.dropna(
        subset=["proposed_start", "actual_start"], how="all"
    )

    # Final sort
    final_df = final_df.sort_values(
        by=["student", "proposed_start", "actual_start"]
    ).reset_index(drop=True)

    # Save output
    final_df.to_csv(output_path, index=False)
    print(f"Successfully generated clean target sheet at: {output_path}")


In [15]:
# --- Execution ---
if __name__ == "__main__":
    # Update these paths to point to your actual local files
    CSV_FILE = "schedules.csv"
    XLSX_FILE = "reports.xlsx"  # Changes automatically to pd.read_excel if you use a true .xlsx file
    OUTPUT_FILE = "merged_target.csv"

    merge_schedules_and_reports(CSV_FILE, XLSX_FILE, OUTPUT_FILE)

Successfully generated clean target sheet at: merged_target.csv
